# Sesgos en ML: Datos ideales vs. Datos desbalanceados

## Introducción
En este notebook se muestran **dos ejemplos realistas**:

1. **Datos ideales**: el dataset está perfectamente balanceado por clases y grupos, pero el modelo introduce sesgo algorítmico al optimizar una métrica global.
2. **Datos desbalanceados**: el dataset presenta una fuerte desproporción en las clases, y se corrige usando técnicas como `class_weight` y `sample_weight`.

El objetivo es que los alumnos vean que los sesgos en ML pueden provenir tanto de los **datos** como de las **decisiones del algoritmo**.

---

## Parte 1: Datos ideales + sesgo algorítmico

In [10]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Dataset balanceado
X, y = make_classification(
    n_samples=4000,
    n_features=10,
    n_informative=5,
    weights=[0.5, 0.5],  # Balanceado
    random_state=42
)

# Creamos un atributo sensible (grupo A o B)
np.random.seed(42)
groups = np.random.choice(["A", "B"], size=len(y), p=[0.5, 0.5])

# Entrenamos un modelo simple
clf = LogisticRegression(max_iter=200)
clf.fit(X, y)

# Predicciones
y_pred = clf.predict(X)

# Hacer classification_report con el conjunto de entrenamiento es un paso inicial para inspeccionar el ajuste, pero para una evaluación realista y validación del modelo es 
# fundamental realizar la evaluación en un conjunto de datos separado
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.80      0.80      1995
           1       0.80      0.80      0.80      2005

    accuracy                           0.80      4000
   macro avg       0.80      0.80      0.80      4000
weighted avg       0.80      0.80      0.80      4000



In [3]:
# Evaluación por grupo
for g in np.unique(groups):
    idx = groups == g
    print(f"Grupo {g}:")
    print(classification_report(y[idx], y_pred[idx]))

Grupo A:
              precision    recall  f1-score   support

           0       0.80      0.79      0.79      1002
           1       0.79      0.80      0.79       995

    accuracy                           0.79      1997
   macro avg       0.79      0.79      0.79      1997
weighted avg       0.79      0.79      0.79      1997

Grupo B:
              precision    recall  f1-score   support

           0       0.80      0.81      0.80       993
           1       0.81      0.80      0.80      1010

    accuracy                           0.80      2003
   macro avg       0.80      0.80      0.80      2003
weighted avg       0.80      0.80      0.80      2003



Observación: Aunque los datos están balanceados, puede que el modelo obtenga mejores métricas en un grupo que en otro → **sesgo algorítmico**.

## Parte 2: Datos desbalanceados + mitigación

In [4]:
# Dataset desbalanceado
X_imb, y_imb = make_classification(
    n_samples=4000,
    n_features=10,
    n_informative=5,
    weights=[0.95, 0.05],  # Fuerte desbalance
    random_state=42
)

print("Distribución de clases:", np.bincount(y_imb))

Distribución de clases: [3781  219]


### Modelo sin corrección

In [5]:
clf_imb = LogisticRegression(max_iter=200)
clf_imb.fit(X_imb, y_imb)
y_pred_imb = clf_imb.predict(X_imb)

print(classification_report(y_imb, y_pred_imb))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97      3781
           1       0.67      0.19      0.29       219

    accuracy                           0.95      4000
   macro avg       0.81      0.59      0.63      4000
weighted avg       0.94      0.95      0.94      4000



⚠️ Aquí el modelo casi ignora la clase minoritaria (recall muy bajo).

### Mitigación con `class_weight="balanced"`

In [6]:
# Cada clase recibe un peso inversamente proporcional a su frecuencia -> Al entrenar, cada error en una clase minoritaria penaliza más que las mayoritarias
# Es posible especificar manualmente los pesos para cada clase, por ejemplo: class_weight={0: 1, 1: 5}
clf_bal = LogisticRegression(max_iter=200, class_weight="balanced")
clf_bal.fit(X_imb, y_imb)
y_pred_bal = clf_bal.predict(X_imb)

print(classification_report(y_imb, y_pred_bal))

              precision    recall  f1-score   support

           0       0.99      0.81      0.89      3781
           1       0.19      0.79      0.31       219

    accuracy                           0.81      4000
   macro avg       0.59      0.80      0.60      4000
weighted avg       0.94      0.81      0.86      4000



Mejora el recall en la clase minoritaria.

### Mitigación con `sample_weight`

In [8]:
# Asignamos pesos manuales inversamente proporcionales a la frecuencia de cada clase
from sklearn.utils.class_weight import compute_sample_weight

# se obtiene un peso por muestra (se podría llegar a personalizar uno a uno)
weights = compute_sample_weight(class_weight="balanced", y=y_imb)

clf_sw = LogisticRegression(max_iter=200)
clf_sw.fit(X_imb, y_imb, sample_weight=weights)

y_pred_sw = clf_sw.predict(X_imb)

print(classification_report(y_imb, y_pred_sw))

              precision    recall  f1-score   support

           0       0.99      0.81      0.89      3781
           1       0.19      0.79      0.31       219

    accuracy                           0.81      4000
   macro avg       0.59      0.80      0.60      4000
weighted avg       0.94      0.81      0.86      4000



In [ ]:
# Asignamos pesos manuales inversamente proporcionales a la frecuencia de cada clase

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X_imb, y_imb, groups, test_size=0.3, stratify=y, random_state=0)

# se obtiene un peso por muestra (se podría llegar a personalizar uno a uno)
weights2 = np.where(g_train=="A", 5, 1)

clf_sw = LogisticRegression(max_iter=200)
clf_sw.fit(X_train, y_train, sample_weight=weights2)

y_pred_sw = clf_sw.predict(X_imb)

print(classification_report(y_imb, y_pred_sw))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97      3781
           1       0.66      0.18      0.29       219

    accuracy                           0.95      4000
   macro avg       0.81      0.59      0.63      4000
weighted avg       0.94      0.95      0.94      4000



Con `sample_weight` logramos un efecto similar a `class_weight`, pero con mayor flexibilidad (se podrían ponderar por grupo, calidad del dato, etc.).

## Conclusiones

- **Parte 1** mostró cómo, incluso con datos perfectamente balanceados, el **modelo puede introducir sesgos** al optimizar métricas globales.
- **Parte 2** mostró cómo el **desbalance de datos genera sesgos** y cómo se pueden mitigar con técnicas de ponderación.

➡️ Moraleja: siempre debemos evaluar los modelos no solo de forma global, sino también por **clase y por grupo sensible**, y aplicar mitigaciones adecuadas según la fuente del sesgo.